# Milestone 3 lab, notebook 7 of 8: The order that vanished

## Where we are

| Notebook | What you do |
|---|---|
| 1. One order, one row | Write the same rows three ways: append, overwrite, upsert |
| 2. Which column is the key? | Merge on the wrong key. Then meet an order that arrives twice in one run |
| 3. Run it twice | Prove that a second run changes nothing |
| 4. At least once, exactly once | Crash the load in three places, and see what each one costs |
| 5. The update that came late | Find a change the watermark has passed. Catch it with a lookback |
| 6. Fix one day | Break Friday's numbers, and rebuild Friday alone |
| **7. The order that vanished** (you are here) | A deleted order, and why the query cannot see it |
| 8. The production version | The three properties, as tests |

Every problem so far was about a row that **changed**. This one is about a row that is **gone**. On Saturday 13 June someone at Arden removes a test order from the order system.

**Part of this notebook is a demonstration.** Step 3 uses a feature of the fake API to stand in for a tool this project does not build.

In [1]:
from lab_helpers import api, lab_warehouse, money, read_orders, run_pipeline, set_morning, show_doc, table, true_day_totals

from daily_orders.report import daily_report, day_totals

## Step 1: A deleted order, seen from the pipeline

Load the mornings up to Saturday 13 June. Then move to Sunday morning, but do not run the pipeline yet.

In [2]:
con = lab_warehouse(fresh=True)
run_pipeline(con, "2026-06-11", "2026-06-12", "2026-06-13")
set_morning("2026-06-14")

report, truth = day_totals(daily_report(con)), true_day_totals()
day = "2026-06-12"
print(f"Friday 12 June in the warehouse: {report[day]['orders']} orders, {money(report[day]['revenue_cents'])}")
print(f"Friday 12 June in the source:    {truth[day]['orders']} orders, {money(truth[day]['revenue_cents'])}")

It is now Sunday 2026-06-14 07:00:00 in Arden.


Friday 12 June in the warehouse: 26 orders, $704.74
Friday 12 June in the source:    25 orders, $690.50


The warehouse has one order more than the source. Which one?

In [3]:
in_warehouse = {row[0] for row in con.execute("SELECT order_id FROM core.orders WHERE order_date = '2026-06-12'")}
set_morning("2026-06-14")
in_source = {order["order_id"] for order in read_orders()[0]}

gone = sorted(in_warehouse - in_source)
print("In the warehouse but not in the source:", gone)
print("Asking the API for it directly:", api.get(f"/orders/{gone[0]}").status_code, api.get(f"/orders/{gone[0]}").json()["detail"])

It is now Sunday 2026-06-14 07:00:00 in Arden.


In the warehouse but not in the source: [1356]
Asking the API for it directly: 404 Order 1356 does not exist.


In [4]:
table(con, "SELECT order_id, customer_id, status, total_cents, order_date, last_run_id FROM core.orders WHERE order_id = %s", [gone[0]])

,order_id,customer_id,status,total_cents,order_date,last_run_id
0,1356,9999,placed,1424,2026-06-12,2026-06-13_1


Customer 9999 is a test account. The order was real enough to be loaded on Saturday morning. Then it was deleted. That is a **hard delete**: the row is removed for good, and no trace is left.

## Step 2: Why the query cannot see it

Your load asks: "which orders have an `updated_at` at or after my watermark?" Ask it, as widely as you like.

In [5]:
set_morning("2026-06-14")
changed, _ = read_orders(updated_since="2026-06-01T00:00:00Z")      # everything since the very first day

print("Orders the query returns:", len(changed))
print(f"Is order {gone[0]} among them?", gone[0] in {order["order_id"] for order in changed})

It is now Sunday 2026-06-14 07:00:00 in Arden.


Orders the query returns: 398
Is order 1356 among them? False


It is not there, and it never will be. A deleted row has no `updated_at`, because it has no row.

> You cannot query a row that does not exist.

This is the limit of **query-based CDC**, which you named in milestone 2. It sees the latest state of rows that exist. It does not see deletes.

## Step 3: A demonstration: the source's own list of changes

A database writes every change to a log before it changes a table: this row was added, this one changed, this one removed. A tool that reads that log sees deletes too. That is **log-based CDC**. Debezium is a well-known open-source tool for it.

This project does not set up a database log. As a stand-in, the fake order system has a **change feed**: an ordered list of every insert, update and delete. Read the last few entries.

In [6]:
changes, position = [], 0
while True:
    body = api.get("/changes", params={"after_pos": position, "limit": 500}).json()
    changes += body["data"]
    position = body["last_pos"]
    if not body["more"]:
        break

print(f"The change feed holds {len(changes)} changes so far. These are the ones for order {gone[0]}:\n")
for change in changes:
    if change["order_id"] == gone[0]:
        before, after = change["before"], change["after"]
        print(f"pos {change['pos']}  op {change['op']}  at {change['changed_at']}"
              f"   before: {before['status'] + ', ' + money(before['total_cents']) if before else '(empty)':<16}"
              f"   after: {after['status'] + ', ' + money(after['total_cents']) if after else '(empty)'}")

The change feed holds 1582 changes so far. These are the ones for order 1356:

pos 1419  op c  at 2026-06-12T19:00:00Z   before: (empty)            after: placed, $14.24
pos 1507  op d  at 2026-06-13T16:00:00Z   before: placed, $14.24     after: (empty)


Two entries for this order. `op c` is the create. `op d` is the delete, and it still knows what the row looked like **before**.

| Field | Meaning |
|---|---|
| `pos` | The position in the list. It gives the true order of the changes |
| `op` | `c` for create, `u` for update, `d` for delete |
| `before`, `after` | The row before and after the change. `after` is empty for a delete |

There is a third way to capture changes, which this project only names: **trigger-based CDC**. The database runs a small piece of code on every change and writes it to a change table. It sees deletes too, but it adds work to every write in the source.

## Step 4: What to do with a delete

You now know order 1356 was deleted. Two choices:

| | Hard delete | Soft delete |
|---|---|---|
| What you do | Remove the row from the warehouse too | Keep the row, and mark it |
| Can you say later what was deleted, and when? | No | Yes |
| What readers must do | Nothing | Skip rows marked as deleted |

Finance must be able to explain why Friday's number moved. So Arden's warehouse uses a **soft delete**. Do it by hand.

In [7]:
with con.transaction():
    con.execute("UPDATE core.orders SET is_deleted = true, deleted_at = '2026-06-13T16:00:00Z' WHERE order_id = %s AND NOT is_deleted", [gone[0]])

table(con, "SELECT order_id, status, total_cents, is_deleted, deleted_at FROM core.orders WHERE order_id = %s", [gone[0]])

,order_id,status,total_cents,is_deleted,deleted_at
0,1356,placed,1424,True,2026-06-13 16:00:00+00:00


The row is still there. Now rebuild Friday's report. The report skips rows where `is_deleted` is true.

In [8]:
from datetime import date

from daily_orders.report import rebuild_days

rebuild_days(con, [date(2026, 6, 12)])

report = day_totals(daily_report(con))
print(f"Friday 12 June in the warehouse: {report[day]['orders']} orders, {money(report[day]['revenue_cents'])}")
print(f"Friday 12 June in the source:    {truth[day]['orders']} orders, {money(truth[day]['revenue_cents'])}")

Friday 12 June in the warehouse: 25 orders, $690.50
Friday 12 June in the source:    25 orders, $690.50


## Step 5: The pipeline does it by itself

The production pipeline reads the change feed on every run, keeps the deletes, and marks them. Start again and let it work.

In [9]:
con.close()
con = lab_warehouse(fresh=True)
run_pipeline(con, "2026-06-11", "2026-06-12", "2026-06-13")
result = run_pipeline(con, "2026-06-14")

from daily_orders.report import format_answer
print(format_answer(result.answer))

Arden daily orders. Report for the morning of Sun 2026-06-14

  Yesterday   Sat 2026-06-13    26 orders      $702.57
  Day before  Fri 2026-06-12    25 orders      $690.50
  Change                         +1 orders      +$12.07

  Earlier days that changed in this run:
    Fri 2026-06-12   -1 orders   -$14.24
      because order 1356: deleted from the order system


The report names the day that moved, by how much, and why. And marking an order that is already marked changes nothing, so this step is safe to run twice too.

In [10]:
again = run_pipeline(con, "2026-06-14")
print("Second run: orders deleted =", again.orders_deleted, " earlier days changed =", again.answer["earlier_changes"])

Second run: orders deleted = 0  earlier days changed = []


In [11]:
show_doc("LLD.md", "6. Deletes")

**From `docs/LLD.md`**

> ## 6. Deletes
> 
> A deleted order has no row, so it has no `updated_at`. The query can never return it. The pipeline reads the order system's **change feed** and keeps the deletes.
> 
> ```sql
> UPDATE core.orders AS t
>    SET is_deleted = true, deleted_at = d.deleted_at, last_run_id = :run_id
>   FROM <the deletes> AS d
>  WHERE t.order_id = d.order_id AND NOT t.is_deleted;     -- marking twice changes nothing
> ```
> 
> | | Hard delete | Soft delete (chosen) |
> |---|---|---|
> | What happens to the row | Removed for good | Kept, with `is_deleted = true` |
> | Can you say later what was deleted? | No | Yes |
> | What readers must do | Nothing | Skip deleted rows |
> 
> The change feed is a stand-in. In a real company this list comes from the source database's own log, through a tool such as Debezium. That is **log-based CDC**.

In [12]:
con.close()

## New words

| Word | Meaning | Here |
|---|---|---|
| **Hard delete** | "A hard delete removes the row for good." | What the order system did to order 1356 |
| **Soft delete** | "A soft delete keeps the row and marks it with a flag." | `is_deleted = true` in the clean orders table |
| **Log-based CDC** | Capturing changes by reading the database's own log. It sees every change, and deletes | Demonstrated here with the fake API's change feed |

The definitions are from the D8Loop chapters [Loading & Write Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/loading-and-write-patterns) and [Change Data Capture (CDC)](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/change-data-capture-cdc).

## What you now know

- A hard delete in the source leaves nothing for an `updated_at` query to find.
- Log-based CDC reads the source's own list of changes, and that list includes deletes.
- A soft delete keeps the row and marks it, so the change can still be explained.
- What this project uses is a stand-in for a log-based tool, and it says so.

**Say it in an interview:** "Query-based CDC cannot see deletes: a deleted row has no `updated_at`. A test order deleted from the source stayed in my warehouse and inflated one day. The real fix is log-based CDC, reading the database's log with something like Debezium. I applied the delete as a soft delete, so the row stays and I can still explain the change."

Next: [notebook 8](08_the_production_version.ipynb) shows the three properties as tests.